# Phase 4 — Hierarchical Memory

**Goal**: Until now, every question was answered in isolation — the agent had no idea what you asked a moment ago. Ask *"What about 2011?"* and it has no clue what *"2011"* refers to. This phase gives the system **memory**, so a conversation actually holds together.

"Hierarchical" memory has **two tiers**:

| Tier | Holds | Lifespan |
|---|---|---|
| **Short-term** | the recent conversation turns | this session; oldest turns get compacted away |
| **Long-term** | durable facts saved to a file | forever — survives restarts |

## Key concepts you'll learn

| Concept | Plain-English |
|---|---|
| **Short-term memory** | A buffer of the last few conversation turns. Bounded — it can't grow forever. |
| **Compaction** | When the buffer is full, the oldest turns get squeezed into a short summary instead of being kept word-for-word. |
| **Memory-augmented prompt** | Before sending a question to the agent, you prepend the conversation so far — that's what makes follow-ups work. |
| **Long-term memory** | Facts written to disk. Reload the program and they're still there. |

## Acceptance criteria
1. A follow-up question ("what about 2011?") is answered **correctly** because memory supplied the context
2. The short-term memory holds the conversation turns after the run
3. `Trace` exported to `traces/traces.jsonl`

## 1. Setup

In [ ]:
# Make the orchestrator/ library importable from notebooks/
import sys, os
from pathlib import Path

# Walk up to the orchestrator root (the folder containing 'data/retail.parquet').
# Idempotent: safe to re-run this cell any number of times.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "retail.parquet").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "data" / "retail.parquet").exists(), f"Could not find orchestrator root from {Path.cwd()}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("CLAUDE_CODE_OAUTH_TOKEN") or os.getenv("ANTHROPIC_API_KEY"), \
    "No auth token found in .env. Run `claude setup-token` and paste into .env."
print(f"[OK] Working dir: {os.getcwd()}")
print(f"[OK] Dev model: {os.getenv('CLAUDE_DEV_MODEL')}")

In [ ]:
import pandas as pd

from orchestrator.tools import get_retail_df
from orchestrator import tools
from orchestrator.observability import Trace, Timer
from orchestrator.agents import AgentRun, run_agent
# Phase 4 adds the memory module:
from orchestrator.memory import ShortTermMemory, LongTermMemory
from claude_agent_sdk import tool, create_sdk_mcp_server, ClaudeAgentOptions

DEV_MODEL = os.getenv("CLAUDE_DEV_MODEL", "claude-haiku-4-5-20251001")

df = get_retail_df()
print(f"Loaded {len(df):,} rows")
print(f"Dev model: {DEV_MODEL}")

## 2. Ground truth

The conversation: first you ask about **2010**, then you ask the follow-up **"what about 2011?"**. So the answer we check against is the **2011** top-3 countries. Compute it with pandas.

In [ ]:
# Top 3 countries by revenue in 2011 - the follow-up question's correct answer
df_2011 = df[(df["Quantity"] > 0) & (df["InvoiceDate"].dt.year == 2011)]

ground_truth = (
    df_2011.groupby("Country")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .reset_index()
)
print("Top 3 countries by revenue, 2011:")
print(ground_truth)

expected_countries = sorted(ground_truth["Country"])
print()
print("Expected countries in the follow-up answer:", expected_countries)

## 3. Rebuild the retail tool + MCP server (recap)

Same `query_retail` tool. Just run it.

In [ ]:
@tool(
    "query_retail",
    description=(
        "Query the retail transactions dataset to return the top N entries "
        "ranked by a metric. Use this for any 'top N' question about products, "
        "countries, or customers. Arguments: year (optional), country (optional "
        "- OMIT to include all countries), top_n (default 10), group_by (one of "
        "'StockCode', 'Country', 'Customer ID'), metric (one of 'revenue', "
        "'Quantity'). Returns a list of dicts, one row each."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "year":     {"type": "integer", "description": "Calendar year filter, e.g. 2010"},
            "country":  {"type": "string",  "description": "Optional country filter. OMIT to include all countries."},
            "top_n":    {"type": "integer", "description": "How many rows to return"},
            "group_by": {"type": "string",  "description": "'StockCode', 'Country', or 'Customer ID'"},
            "metric":   {"type": "string",  "description": "'revenue' or 'Quantity'"},
        },
        "required": [],
    },
)
async def query_retail_tool(args):
    """Adapter: the agent passes args as a dict; we call the pandas function."""
    rows = tools.query_retail(
        year=args.get("year"),
        country=args.get("country"),
        top_n=args.get("top_n", 10),
        group_by=args.get("group_by", "StockCode"),
        metric=args.get("metric", "revenue"),
    )
    import json
    return {"content": [{"type": "text", "text": json.dumps(rows, default=str)}]}


retail_server = create_sdk_mcp_server(name="retail", version="1.0.0", tools=[query_retail_tool])
print("[OK] Retail tool + MCP server ready")

## 4. The DataAgent (recap)

Same worker as before. Just run it.

In [ ]:
data_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt=(
        "You are a retail data analyst. You may be given a conversation so "
        "far, followed by a new question. Use the conversation to understand "
        "what the new question refers to (for example, a follow-up like "
        "'what about 2011?'). Answer using the query_retail tool, and present "
        "the result as a short, clear markdown table. "
        "Never invent numbers - report exactly what the tool returns."
    ),
    mcp_servers={"retail": retail_server},
    allowed_tools=["mcp__retail__query_retail"],
    max_turns=5,
)
print("[OK] DataAgent configured")

## 5. Short-term memory — the conversation buffer

`ShortTermMemory` (in `orchestrator/memory.py`) is a small buffer that holds the last few conversation turns. Two methods you'll use:

- `memory.add(role, text)` — record one turn (`role` is `"user"` or `"assistant"`)
- `memory.as_context()` — render the whole conversation as text, to prepend to a prompt

Run the cell below to see it work — no TODO here, just watch.

In [ ]:
demo_mem = ShortTermMemory(max_turns=6)
demo_mem.add("user", "What were the top 3 countries by revenue in 2010?")
demo_mem.add("assistant", "United Kingdom, EIRE, and Netherlands.")

print("----- memory.as_context() -----")
print(demo_mem.as_context())
print()
print(f"turns held: {len(demo_mem)}")

## 6. Why memory matters — a follow-up with no context

Here's the problem memory solves. We ask the DataAgent *"What about 2011?"* with **no memory** — no conversation history. Watch it get confused: it has no idea what "2011" is about.

In [ ]:
no_memory_question = "What about 2011?"
demo_run = await run_agent("DataAgent", no_memory_question, data_options)

print("----- Agent answer to a context-free follow-up -----")
print(demo_run.answer)
print()
print("^ Notice: with no memory, the agent cannot tell what '2011' refers to.")

## 7. Answer with memory

`answer_with_memory()` does three things:

1. Build a prompt = **the conversation so far** + the new question
2. Run the DataAgent on that prompt
3. Save this exchange (question + answer) back into memory

Steps 1 and 3 are yours.

**TODO 1**: build the memory-augmented prompt.
**TODO 2**: save the exchange into memory.

In [ ]:
async def answer_with_memory(question, memory, data_options):
    """Answer a question with the conversation so far prepended as context."""

    # TODO 1 (filled) - build the prompt with memory context.
    # The DataAgent sees the conversation so far BEFORE the new question,
    # so a follow-up like "what about 2011?" has something to refer to.
    prompt = (
        "Conversation so far:\n"
        + memory.as_context()
        + "\n\nNew question: "
        + question
    )

    data_run = await run_agent("DataAgent", prompt, data_options)
    answer = data_run.answer

    # TODO 2 (filled) - save this exchange into memory, so the NEXT
    # question can see it. Two turns: the user's question, then the answer.
    memory.add("user", question)
    memory.add("assistant", answer)

    return {"answer": answer, "run": data_run}

print("[OK] answer_with_memory defined")

## 8. Run the conversation

Two turns. **Turn 1** is a full question about 2010 — run for you below. **Turn 2** is your follow-up.

**TODO 3**: write the follow-up question. It must be a *short follow-up* that only makes sense *with* memory — something like "what about 2011?" — NOT a full self-contained question.

In [ ]:
memory = ShortTermMemory(max_turns=6)

# --- Turn 1: a full question about 2010 (given) ---
q1 = "What were the top 3 countries by revenue in 2010?"
result1 = await answer_with_memory(q1, memory, data_options)
print("Q1:", q1)
print(result1["answer"][:300], "...")
print()

# --- Turn 2: your follow-up ---
# TODO 3 (filled) - a short follow-up that only makes sense WITH memory.
# On its own "What about 2011?" is meaningless; memory supplies the rest.
QUESTION = "What about 2011?"

trace = Trace(model=DEV_MODEL, question=QUESTION)
with Timer() as t:
    result2 = await answer_with_memory(QUESTION, memory, data_options)

# roll both turns' usage into the trace
for r in (result1["run"], result2["run"]):
    trace.input_tokens        += r.input_tokens
    trace.output_tokens       += r.output_tokens
    trace.cached_input_tokens += r.cached_input_tokens
    trace.n_tool_calls        += r.n_tool_calls
trace.n_delegations = 2
trace.latency_ms    = t.elapsed_ms
trace.answer        = result2["answer"]
trace.compute_cost()

print("Q2 (follow-up):", QUESTION)
print(result2["answer"])

## 9. Long-term memory — facts that survive a restart

`ShortTermMemory` forgets old turns. `LongTermMemory` writes facts to a JSON file on disk — they're still there after you restart the kernel or close the laptop.

Run the cell below to see it. No TODO — just watch.

In [ ]:
ltm = LongTermMemory(path="memory/long_term.json")
ltm.remember("The user is interested in country-level revenue analysis.")
ltm.remember("In 2010 the United Kingdom led retail revenue by a wide margin.")

print("Facts saved to memory/long_term.json:")
for fact in ltm.recall():
    print(f"  - {fact}")

# Prove it persisted: a brand-new LongTermMemory reads the same file
reloaded = LongTermMemory(path="memory/long_term.json")
print()
print(f"A fresh LongTermMemory reloads {len(reloaded.recall())} fact(s) from disk.")

## 10. Verify (acceptance criteria)

In [ ]:
# TODO 4 (filled) - build `missing`: the ground-truth countries whose name
# does NOT appear in the follow-up answer. Empty list == every one is present.
missing = []
for country in expected_countries:
    if country not in result2["answer"]:
        missing.append(country)

trace.passed = (missing == [])
print(f"Missing countries: {missing}")
print(f"Passed: {trace.passed}")
assert trace.passed, f"Follow-up answer missed: {missing} (did memory carry the context?)"

# Phase 4 acceptance criteria
assert len(memory) >= 2, f"Expected memory to hold the conversation, got {len(memory)} turns"

trace.append_jsonl()
print("\n[OK] Acceptance criteria met - memory carried the context, trace saved")

## 11. Quiz (~5 min — answer in a new markdown cell)

**TODO 5**: Answer in your own words.

1. **Why memory**: in Phases 1-3 every question stood alone. Why does the follow-up "what about 2011?" *fail* without memory? What exactly does the memory-augmented prompt give the agent?

2. **Short vs long term**: `ShortTermMemory` forgets old turns; `LongTermMemory` keeps facts on disk forever. Give one thing that belongs in short-term memory and one that belongs in long-term — and why.

3. **Compaction**: `ShortTermMemory` has a `max_turns` limit. What happens to the oldest turn when the buffer is full? Why not just keep *every* turn forever (think about prompt size and cost)?

4. **The prompt grows**: every turn, the prompt includes the whole conversation so far. What does that do to `input_tokens` and cost as a conversation gets longer? How does compaction help?

5. **Statelessness**: the LLM itself has no memory between calls — each `run_agent` call is fresh. So where does the "memory" in this phase actually live? (Hint: it's not in the model.)

### TODO 5 — Quiz answers (model answers — read these)

> These are filled-in model answers. Read them as the lesson. For your Medium/LinkedIn writeup, rephrase the ones you find interesting in your own words.

**1. Why memory.** On its own, *"What about 2011?"* is a fragment — the agent can't tell what "2011" refers to (revenue? countries? products?). The memory-augmented prompt prepends the earlier turn, so the agent sees the 2010 question *and answer* first and can infer the follow-up means "the same thing — top 3 countries by revenue — but for 2011."

**2. Short vs long term.** Short-term: the current conversation turns (e.g. "the user asked about 2010, then 2011") — relevant only to this session, noise afterwards. Long-term: a durable fact like "this user always wants country-level analysis" — useful across sessions, so it should survive a restart.

**3. Compaction.** When the buffer hits `max_turns`, the oldest turn is dropped from the live list and folded into a short summary. You can't keep every turn forever because the *whole* conversation is re-sent in the prompt on every call — an unbounded conversation means an ever-growing prompt, rising cost, and eventually blowing past the model's context limit.

**4. The prompt grows.** Each turn carries the whole conversation, so `input_tokens` climbs every turn and cost climbs with it (you pay per input token). Compaction caps it: old turns become a short summary instead of full text, so the prompt stays roughly bounded instead of growing without limit.

**5. Statelessness.** The LLM has no memory between calls — every `run_agent` call starts fresh. The "memory" lives in *your code*: the `ShortTermMemory` object (and the `long_term.json` file on disk). You re-supply it by prepending it to the prompt each call. The model doesn't remember — your program remembers and re-tells it.

## Phase 4 done when:
- [x] TODO 1 (memory-augmented prompt) filled in
- [x] TODO 2 (save the exchange to memory) filled in
- [x] TODO 3 (your follow-up question) filled in
- [x] TODO 4 (assertion) filled in
- [x] TODO 5 (quiz) answered
- [x] Cell 13 shows the agent confused by a context-free follow-up
- [x] Cell 17 shows the follow-up answered correctly *with* memory
- [x] Notebook runs top-to-bottom without errors
- [x] `traces/traces.jsonl` has a new line

Then ping me — we'll review your work, then move to Phase 5 (RAG with citations).